# Clinical BERT NER Fine-tuning and Testing

This notebook fine-tunes Clinical BERT for Named Entity Recognition (NER) using sentence and tag data.


## 1. Install Required Libraries


In [ ]:
!pip install transformers torch datasets seqeval scikit-learn -q


## 2. Import Libraries


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification
)
from datasets import Dataset as HFDataset
from seqeval.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


## 3. Load MedMentions Dataset from HuggingFace

Load the MedMentions dataset and process it to extract sentences and NER tags.


In [ ]:
from datasets import load_dataset

# Load MedMentions dataset from HuggingFace
# MedMentions is a biomedical NER dataset
print("Loading MedMentions dataset from HuggingFace...")
try:
    # Try the bigbio version first (most common)
    dataset = load_dataset("bigbio/medmentions", "medmentions_full_ner", trust_remote_code=True)
    print("Loaded: bigbio/medmentions (medmentions_full_ner)")
except:
    try:
        # Alternative: try other versions
        dataset = load_dataset("bigbio/medmentions", trust_remote_code=True)
        print("Loaded: bigbio/medmentions (default split)")
    except:
        # Fallback: use a different biomedical NER dataset
        print("MedMentions not available")

print(f"\nDataset splits: {list(dataset.keys())}")
print(f"Dataset features: {dataset[list(dataset.keys())[0]].features}")

# Display sample
print("\nSample from dataset:")
print(dataset[list(dataset.keys())[0]][0])


In [ ]:
# Define label mapping: map original labels to final labels
# All labels not in this dictionary will be mapped to 'O'
keep_labels = {
    # Example mappings - update these based on your needs
    # 'current_label': 'final_label',
    # 'T001': 'DISEASE',
    # 'T002': 'ANATOMY',
    # 'T003': 'PROCEDURE',
    # Add your label mappings here
}

def map_label(original_label, keep_labels_dict):
    """
    Map original label to final label using keep_labels dictionary.
    Returns 'O' if label is not in the dictionary.
    """
    if original_label in keep_labels_dict:
        return keep_labels_dict[original_label]
    else:
        return 'O'


## 4. Process Dataset: Convert to Sentences and NER Tags


In [ ]:
def process_medmentions_to_ner_format(dataset_split, keep_labels_dict):
    """
    Process MedMentions dataset to extract sentences and NER tags in BIO format.
    Handles bigbio/medmentions format with 'passages' and 'entities'.
    Maps labels according to keep_labels_dict, all others become 'O'.
    """
    only_sentences = []
    ner_tags_list = []
    
    for idx, example in enumerate(dataset_split):
        # bigbio/medmentions has 'passages' (text) and 'entities' (annotations)
        if 'passages' in example and 'entities' in example:
            passages = example['passages']
            entities = example['entities']
            
            # Process each passage (document may have multiple passages)
            for passage in passages:
                if 'text' in passage and len(passage['text']) > 0:
                    text = passage['text'][0] if isinstance(passage['text'], list) else passage['text']
                    
                    # Get passage offsets to map entities correctly
                    passage_offsets = passage.get('offsets', [])
                    if passage_offsets:
                        passage_start = passage_offsets[0][0] if isinstance(passage_offsets[0], list) else passage_offsets[0]
                    else:
                        passage_start = 0
                    
                    # Tokenize text into words
                    tokens = text.split()
                    tags = ['O'] * len(tokens)
                    
                    # Map entities to this passage
                    for entity in entities:
                        if 'offsets' in entity and 'type' in entity:
                            offsets = entity['offsets']
                            entity_type = entity['type']
                            
                            # Map the label using keep_labels_dict
                            mapped_label = map_label(entity_type, keep_labels_dict)
                            
                            # Skip if mapped to 'O'
                            if mapped_label == 'O':
                                continue
                            
                            # Get entity start and end (relative to document)
                            if isinstance(offsets[0], list):
                                entity_start = offsets[0][0]
                                entity_end = offsets[-1][1]
                            else:
                                entity_start = offsets[0]
                                entity_end = offsets[-1]
                            
                            # Check if entity is in this passage
                            passage_end = passage_start + len(text)
                            if entity_start >= passage_start and entity_end <= passage_end:
                                # Convert to passage-relative positions
                                rel_start = entity_start - passage_start
                                rel_end = entity_end - passage_start
                                
                                # Map character positions to token indices
                                char_pos = 0
                                token_start_idx = None
                                token_end_idx = None
                                
                                for i, token in enumerate(tokens):
                                    token_start = char_pos
                                    token_end = char_pos + len(token)
                                    
                                    # Check if token overlaps with entity
                                    if token_start < rel_end and token_end > rel_start:
                                        if token_start_idx is None:
                                            token_start_idx = i
                                        token_end_idx = i
                                    
                                    char_pos = token_end + 1  # +1 for space
                                
                                # Assign BIO tags with mapped label
                                if token_start_idx is not None and token_end_idx is not None:
                                    tags[token_start_idx] = f'B-{mapped_label}'
                                    for j in range(token_start_idx + 1, token_end_idx + 1):
                                        if j < len(tags):
                                            tags[j] = f'I-{mapped_label}'
                    
                    # Only add non-empty passages
                    if len(tokens) > 0:
                        only_sentences.append(text)
                        ner_tags_list.append(tags)
        
        # Fallback: handle other formats
        elif 'tokens' in example:
            tokens = example['tokens']
            if 'ner_tags' in example:
                tags = example['ner_tags']
                if isinstance(tags[0], (int, np.integer)):
                    tags = [str(tag) for tag in tags]
                
                # Map existing tags using keep_labels_dict
                mapped_tags = []
                for tag in tags:
                    if tag == 'O':
                        mapped_tags.append('O')
                    elif tag.startswith('B-'):
                        base_label = tag[2:]
                        mapped_label = map_label(base_label, keep_labels_dict)
                        mapped_tags.append(f'B-{mapped_label}' if mapped_label != 'O' else 'O')
                    elif tag.startswith('I-'):
                        base_label = tag[2:]
                        mapped_label = map_label(base_label, keep_labels_dict)
                        mapped_tags.append(f'I-{mapped_label}' if mapped_label != 'O' else 'O')
                    else:
                        # Not in BIO format, try direct mapping
                        mapped_label = map_label(tag, keep_labels_dict)
                        mapped_tags.append(mapped_label)
                tags = mapped_tags
            else:
                tags = ['O'] * len(tokens)
            
            sentence = ' '.join(tokens)
            only_sentences.append(sentence)
            ner_tags_list.append(tags)
            
        elif 'text' in example:
            text = example['text']
            tokens = text.split()
            tags = ['O'] * len(tokens)
            
            if 'entities' in example:
                entities = example['entities']
                for entity in entities:
                    if 'start' in entity and 'end' in entity:
                        start = entity['start']
                        end = entity['end']
                        label = entity.get('type', entity.get('label', 'ENTITY'))
                        
                        # Map the label using keep_labels_dict
                        mapped_label = map_label(label, keep_labels_dict)
                        
                        # Skip if mapped to 'O'
                        if mapped_label == 'O':
                            continue
                        
                        char_pos = 0
                        token_start_idx = None
                        token_end_idx = None
                        
                        for i, token in enumerate(tokens):
                            token_start = char_pos
                            token_end = char_pos + len(token)
                            
                            if token_start <= start < token_end:
                                token_start_idx = i
                            if token_start < end <= token_end:
                                token_end_idx = i
                                break
                            
                            char_pos = token_end + 1
                        
                        if token_start_idx is not None and token_end_idx is not None:
                            tags[token_start_idx] = f'B-{mapped_label}'
                            for j in range(token_start_idx + 1, token_end_idx + 1):
                                if j < len(tags):
                                    tags[j] = f'I-{mapped_label}'
            
            only_sentences.append(text)
            ner_tags_list.append(tags)
    
    return only_sentences, ner_tags_list

In [ ]:
# Process each split separately (train, validation, test)
print("Available splits:", list(dataset.keys()))

# Process train split
if 'train' in dataset:
    print(f"\nProcessing train split...")
    print(f"Total examples in train split: {len(dataset['train'])}")
    train_sentences, train_tags = process_medmentions_to_ner_format(dataset['train'], keep_labels)
    print(f"Processed {len(train_sentences)} sentences from train split")
else:
    print("Warning: 'train' split not found in dataset")
    train_sentences, train_tags = [], []

# Process validation split
if 'validation' in dataset or 'val' in dataset:
    val_split_name = 'validation' if 'validation' in dataset else 'val'
    print(f"\nProcessing {val_split_name} split...")
    print(f"Total examples in {val_split_name} split: {len(dataset[val_split_name])}")
    val_sentences, val_tags = process_medmentions_to_ner_format(dataset[val_split_name], keep_labels)
    print(f"Processed {len(val_sentences)} sentences from {val_split_name} split")
else:
    print("Warning: 'validation' or 'val' split not found in dataset")
    val_sentences, val_tags = [], []

# Process test split
if 'test' in dataset:
    print(f"\nProcessing test split...")
    print(f"Total examples in test split: {len(dataset['test'])}")
    test_sentences, test_tags = process_medmentions_to_ner_format(dataset['test'], keep_labels)
    print(f"Processed {len(test_sentences)} sentences from test split")
else:
    print("Warning: 'test' split not found in dataset")
    test_sentences, test_tags = [], []

# Display examples
if len(train_sentences) > 0:
    print(f"\nExample from train split:")
    print(f"  Sentence: {train_sentences[0][:100]}...")
    print(f"  Tags (first 20): {train_tags[0][:20]}")


## 5. Define Label Mappings


In [ ]:
# Get all unique tags from all splits
all_tags = set()
for tags in train_tags + val_tags + test_tags:
    all_tags.update(tags)

# Create label mappings
unique_tags = sorted(list(all_tags))
label2id = {tag: idx for idx, tag in enumerate(unique_tags)}
id2label = {idx: tag for tag, idx in label2id.items()}

print(f"Number of unique labels: {len(unique_tags)}")
print(f"Labels: {unique_tags}")
print(f"\nLabel to ID mapping: {label2id}")


In [ ]:
# Calculate class weights to handle label imbalance
from collections import Counter

# Count label frequencies across all splits
label_counts = Counter()
for tags in train_tags + val_tags + test_tags:
    label_counts.update(tags)

# Calculate inverse frequency weights
total_samples = sum(label_counts.values())
class_weights_dict = {}
for label, count in label_counts.items():
    # Inverse frequency weighting: more weight for rare labels
    class_weights_dict[label] = total_samples / (len(label_counts) * count)

# Convert to tensor in the order of label2id
weight_tensor = torch.tensor(
    [class_weights_dict.get(id2label[i], 1.0) for i in range(len(unique_tags))], 
    dtype=torch.float32
).to(DEVICE)

print(f"Class weights calculated for {len(class_weights_dict)} labels")
print(f"Weight range: min={weight_tensor.min():.4f}, max={weight_tensor.max():.4f}")
print(f"\nTop 5 most common labels (lowest weights):")
for label, count in label_counts.most_common(5):
    print(f"  {label}: count={count}, weight={class_weights_dict[label]:.4f}")
print(f"\nTop 5 rarest labels (highest weights):")
for label, count in label_counts.most_common()[-5:]:
    print(f"  {label}: count={count}, weight={class_weights_dict[label]:.4f}")


## 5. Load Clinical BERT Model and Tokenizer


In [ ]:
# Clinical BERT model (you can use other clinical models like emilyalsentzer/Bio_ClinicalBERT)
model_name = "emilyalsentzer/Bio_ClinicalBERT"  # or "dmis-lab/biobert-base-cased-v1.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(unique_tags),
    id2label=id2label,
    label2id=label2id
)

print(f"Model loaded: {model_name}")
print(f"Model has {model.num_parameters():,} parameters")


## 6. Tokenize and Align Labels


In [ ]:
def tokenize_and_align_labels(sentences, tags, tokenizer, label2id, max_length=512):
    """
    Tokenize sentences and align NER tags with tokenized words.
    Handles subword tokenization (e.g., 'diabetes' -> 'diab', '##etes').
    """
    tokenized_inputs = tokenizer(
        sentences,
        truncation=True,
        padding=True,
        max_length=max_length,
        is_split_into_words=False,
        return_tensors=None
    )
    
    labels = []
    for i, tag_sequence in enumerate(tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            # Special tokens get -100 (ignored in loss calculation)
            if word_idx is None:
                label_ids.append(-100)
            # Set label for the first token of each word
            elif word_idx != previous_word_idx:
                # Map tag to label ID
                tag = tag_sequence[word_idx] if word_idx < len(tag_sequence) else "O"
                label_ids.append(label2id.get(tag, label2id["O"]))
            else:
                # For subword tokens, use -100 or keep the same label
                # Option 1: Ignore subword tokens (recommended)
                label_ids.append(-100)
                # Option 2: Keep same label (uncomment if preferred)
                # label_ids.append(label_ids[-1])
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenize each split separately
print("Tokenizing sentences and aligning labels...")

print("Tokenizing train split...")
train_tokenized = tokenize_and_align_labels(train_sentences, train_tags, tokenizer, label2id)
print(f"Tokenized {len(train_tokenized['input_ids'])} train sequences")

if len(val_sentences) > 0:
    print("Tokenizing validation split...")
    val_tokenized = tokenize_and_align_labels(val_sentences, val_tags, tokenizer, label2id)
    print(f"Tokenized {len(val_tokenized['input_ids'])} validation sequences")
else:
    val_tokenized = None

print("Tokenizing test split...")
test_tokenized = tokenize_and_align_labels(test_sentences, test_tags, tokenizer, label2id)
print(f"Tokenized {len(test_tokenized['input_ids'])} test sequences")

print(f"\nExample tokenized input (first 20 tokens): {train_tokenized['input_ids'][0][:20]}")
print(f"Example labels (first 20): {train_tokenized['labels'][0][:20]}")


In [ ]:
# Convert to HuggingFace Dataset format using built-in splits
train_dataset = HFDataset.from_dict(train_tokenized)
test_dataset = HFDataset.from_dict(test_tokenized)

if val_tokenized is not None:
    val_dataset = HFDataset.from_dict(val_tokenized)
    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    print(f"Test samples: {len(test_dataset)}")
else:
    val_dataset = None
    print(f"Training samples: {len(train_dataset)}")
    print(f"Test samples: {len(test_dataset)}")
    print("Note: No validation split available")


## 8. Define Metrics for Evaluation


In [ ]:
def compute_metrics(eval_pred):
    """
    Compute precision, recall, F1, and accuracy for NER task.
    """
    predictions, labels = eval_pred
    
    # Convert predictions to label IDs (argmax)
    predictions = np.argmax(predictions, axis=2)
    
    # Remove ignored index (special tokens)
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    results = {
        "accuracy": accuracy_score(true_labels, true_predictions),
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions)
    }
    
    return results


## 9. Set Up Training Configuration


In [ ]:
# Training hyperparameters
LEARNING_RATE = 5e-5  # Increased for faster convergence
BATCH_SIZE = 16
NUM_EPOCHS = 8  # Increased for more training time
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Training configuration:")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Device: {DEVICE}")


## 10. Create DataLoaders


In [ ]:
# Create DataLoaders
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

print(f"Created DataLoaders: {len(train_loader)} train batches, {len(val_loader)} validation batches, {len(test_loader)} test batches")

# Move model to device
model = model.to(DEVICE)


In [ ]:
# Create weighted loss function for imbalanced classes
criterion = CrossEntropyLoss(weight=weight_tensor, ignore_index=-100)
print("Weighted loss function created with class weights")


In [ ]:
def evaluate_model(model, dataloader, device):
    """Evaluate model and return metrics."""
    model.eval()
    all_predictions = []
    all_labels = []
    total_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            total_loss += loss.item()
            
            # Get predictions
            predictions = torch.argmax(outputs.logits, dim=-1)
            labels = batch['labels']
            
            # Store predictions and labels
            for pred, label in zip(predictions.cpu().numpy(), labels.cpu().numpy()):
                # Remove ignored tokens (-100)
                mask = label != -100
                all_predictions.append([id2label[p] for p, m in zip(pred, mask) if m])
                all_labels.append([id2label[l] for l, m in zip(label, mask) if m])
    
    # Calculate metrics
    metrics = {
        "loss": total_loss / len(dataloader),
        "accuracy": accuracy_score(all_labels, all_predictions),
        "precision": precision_score(all_labels, all_predictions),
        "recall": recall_score(all_labels, all_predictions),
        "f1": f1_score(all_labels, all_predictions)
    }
    
    return metrics, all_predictions, all_labels


## 11. Training Loop


In [ ]:
# Setup optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * NUM_EPOCHS

# Training loop
print("Starting fine-tuning...")
for epoch in range(NUM_EPOCHS):
    # Training phase
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
    
    for batch_idx, batch in enumerate(progress_bar):
        # Move batch to device
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        
        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})
    
    train_avg_loss = total_loss / len(train_loader)
    
    # Validation phase
    model.eval()
    val_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]", leave=False):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            val_loss += loss.item()
    
    val_avg_loss = val_loss / len(val_loader)
    
    # Print epoch results
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss: {train_avg_loss:.4f}, Val Loss: {val_avg_loss:.4f}")

print("\nFine-tuning completed!")


## 12. Evaluation Function


In [ ]:
def evaluate_model(model, dataloader, device):
    """Evaluate model and return metrics."""
    model.eval()
    all_predictions = []
    all_labels = []
    total_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            total_loss += loss.item()
            
            # Get predictions
            predictions = torch.argmax(outputs.logits, dim=-1)
            labels = batch['labels']
            
            # Store predictions and labels
            for pred, label in zip(predictions.cpu().numpy(), labels.cpu().numpy()):
                # Remove ignored tokens (-100)
                mask = label != -100
                all_predictions.append([id2label[p] for p, m in zip(pred, mask) if m])
                all_labels.append([id2label[l] for l, m in zip(label, mask) if m])
    
    # Calculate metrics
    metrics = {
        "loss": total_loss / len(dataloader),
        "accuracy": accuracy_score(all_labels, all_predictions),
        "precision": precision_score(all_labels, all_predictions),
        "recall": recall_score(all_labels, all_predictions),
        "f1": f1_score(all_labels, all_predictions)
    }
    
    return metrics, all_predictions, all_labels

# Evaluate on test set
print("Evaluating on test set...")
test_metrics, test_predictions, test_labels = evaluate_model(model, test_loader, DEVICE)

print("\nTest Set Results:")
for key, value in test_metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")


## 13. Detailed Classification Report


In [ ]:
# Generate classification report using results from evaluation
report = classification_report(test_labels, test_predictions)
print("Detailed Classification Report:")
print(report)


## 14. Save the Fine-tuned Model


In [ ]:
model.save_pretrained("./clinical_bert_ner_model_final")
tokenizer.save_pretrained("./clinical_bert_ner_model_final")

print("Model and tokenizer saved to ./clinical_bert_ner_model_final")


## 15. Test on New Sentences


In [ ]:
def predict_ner(sentence, model, tokenizer, id2label):
    """
    Predict NER tags for a new sentence.
    """
    # Tokenize
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
    
    # Get predictions
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)
    
    # Get word IDs to align predictions with words
    word_ids = inputs.word_ids()
    
    # Map predictions to words
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    predicted_labels = [id2label[pred.item()] for pred in predictions[0]]
    
    # Filter out special tokens and subword tokens
    word_predictions = []
    previous_word_idx = None
    for word_idx, token, label in zip(word_ids, tokens, predicted_labels):
        if word_idx is not None and word_idx != previous_word_idx:
            word_predictions.append((token, label))
        previous_word_idx = word_idx
    
    return word_predictions

# Test on example sentences
test_sentences = [
    "Patient has diabetes and high blood pressure",
    "The medication was prescribed for hypertension"
]

print("Testing on new sentences:\n")
for sentence in test_sentences:
    print(f"Sentence: {sentence}")
    predictions = predict_ner(sentence, model, tokenizer, id2label)
    print("Predictions:")
    for token, label in predictions:
        if not token.startswith('##'):  # Skip subword tokens for display
            print(f"  {token}: {label}")
    print()


## 16. Load Saved Model for Future Use


In [ ]:
# To load the saved model later:
# loaded_model = AutoModelForTokenClassification.from_pretrained("./clinical_bert_ner_model_final")
# loaded_tokenizer = AutoTokenizer.from_pretrained("./clinical_bert_ner_model_final")
# 
# # Use the loaded model for predictions
# predictions = predict_ner("Your sentence here", loaded_model, loaded_tokenizer, id2label)

print("Model loading code provided above (commented out)")


## 17. Evaluate on Custom Test Dataset

Load and evaluate the fine-tuned model on a custom CSV test dataset with 'note' and 'tags' columns.


In [ ]:
import pandas as pd
import json
import ast

# Load custom test dataset
# Update this path to your CSV file location
custom_test_csv = "custom_test_dataset.csv"  # Change this to your CSV file path

print(f"Loading custom test dataset from {custom_test_csv}...")
try:
    df_custom = pd.read_csv(custom_test_csv)
    print(f"Loaded {len(df_custom)} rows")
    print(f"Columns: {df_custom.columns.tolist()}")
    
    # Display first few rows
    print("\nFirst few rows:")
    print(df_custom.head())
except FileNotFoundError:
    print(f"Error: File '{custom_test_csv}' not found. Please update the path.")
    raise
except Exception as e:
    print(f"Error loading CSV: {e}")
    raise

# Verify required columns exist
if 'note' not in df_custom.columns or 'tags' not in df_custom.columns:
    raise ValueError("CSV must contain 'note' and 'tags' columns")

print(f"\nDataset shape: {df_custom.shape}")


In [ ]:
# Parse tags column (JSON string format)
def parse_tags(tags_str):
    """Parse tags from JSON string format."""
    if pd.isna(tags_str):
        return []
    try:
        # Try JSON parsing first
        return json.loads(tags_str)
    except (json.JSONDecodeError, TypeError):
        try:
            # Fallback to ast.literal_eval for Python list strings
            return ast.literal_eval(tags_str)
        except (ValueError, SyntaxError):
            print(f"Warning: Could not parse tags: {tags_str[:50]}...")
            return []

# Extract notes and tags
custom_notes = []
custom_tags = []
skipped_rows = []

for idx, row in df_custom.iterrows():
    note = str(row['note']) if pd.notna(row['note']) else ""
    tags = parse_tags(row['tags'])
    
    # Split note into words
    words = note.split()
    
    # Verify word count matches tag count
    if len(words) != len(tags):
        print(f"Warning: Row {idx}: word count ({len(words)}) != tag count ({len(tags)}). Skipping.")
        skipped_rows.append(idx)
        continue
    
    if len(words) == 0:
        print(f"Warning: Row {idx}: empty note. Skipping.")
        skipped_rows.append(idx)
        continue
    
    custom_notes.append(note)
    custom_tags.append(tags)

print(f"\nProcessed {len(custom_notes)} valid examples")
if skipped_rows:
    print(f"Skipped {len(skipped_rows)} rows due to mismatches")

# Display example
if len(custom_notes) > 0:
    print(f"\nExample from custom dataset:")
    print(f"  Note: {custom_notes[0][:100]}...")
    print(f"  Tags (first 20): {custom_tags[0][:20]}")


In [ ]:
# Load saved model and tokenizer
model_path = "./clinical_bert_ner_model_final"

print(f"Loading saved model from {model_path}...")
try:
    loaded_model = AutoModelForTokenClassification.from_pretrained(model_path)
    loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # Extract label mappings from model config
    loaded_id2label = loaded_model.config.id2label
    loaded_label2id = loaded_model.config.label2id
    
    print(f"Model loaded successfully!")
    print(f"Number of labels: {len(loaded_id2label)}")
    print(f"Labels: {sorted(loaded_id2label.values())}")
    
    # Move model to device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loaded_model = loaded_model.to(DEVICE)
    print(f"Model moved to {DEVICE}")
    
except Exception as e:
    print(f"Error loading model: {e}")
    print("Make sure you have run the training and saved the model first.")
    raise


In [ ]:
# Check label compatibility and handle unknown labels
print("Checking label compatibility...")

# Get all unique tags from custom dataset
custom_unique_tags = set()
for tags in custom_tags:
    custom_unique_tags.update(tags)

print(f"Unique tags in custom dataset: {len(custom_unique_tags)}")
print(f"Tags: {sorted(custom_unique_tags)}")

# Check for unknown labels (not in model's label set)
unknown_labels = custom_unique_tags - set(loaded_label2id.keys())
if unknown_labels:
    print(f"\nWarning: Found {len(unknown_labels)} unknown labels not in model's label set:")
    print(f"  Unknown labels: {sorted(unknown_labels)}")
    print("  These will be mapped to 'O' during tokenization")
else:
    print("\nAll labels in custom dataset are present in model's label set!")

# Verify all required labels exist in model
required_labels = set(loaded_label2id.keys())
print(f"\nModel supports {len(required_labels)} labels")


In [ ]:
# Tokenize custom dataset using the same function
print("Tokenizing custom test dataset...")

# Filter out unknown labels by mapping them to 'O'
def filter_tags(tags, label2id):
    """Map unknown labels to 'O'."""
    filtered = []
    for tag in tags:
        if tag in label2id:
            filtered.append(tag)
        else:
            filtered.append('O')
    return filtered

# Apply label filtering
custom_tags_filtered = [filter_tags(tags, loaded_label2id) for tags in custom_tags]

# Tokenize using existing function
custom_tokenized = tokenize_and_align_labels(custom_notes, custom_tags_filtered, loaded_tokenizer, loaded_label2id)

print(f"Tokenized {len(custom_tokenized['input_ids'])} sequences")
print(f"\nExample tokenized input (first 20 tokens): {custom_tokenized['input_ids'][0][:20]}")
print(f"Example labels (first 20): {custom_tokenized['labels'][0][:20]}")


In [ ]:
# Create DataLoader for custom test dataset
custom_dataset = HFDataset.from_dict(custom_tokenized)

data_collator = DataCollatorForTokenClassification(tokenizer=loaded_tokenizer)
BATCH_SIZE = 16  # Use same batch size as training

custom_loader = DataLoader(
    custom_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

print(f"Created DataLoader: {len(custom_loader)} batches")
print(f"Total samples: {len(custom_dataset)}")


In [ ]:
# Evaluate model on custom test dataset
print("Evaluating model on custom test dataset...")

# Use the existing evaluate_model function
custom_metrics, custom_predictions, custom_labels = evaluate_model(loaded_model, custom_loader, DEVICE)

print("\n" + "="*50)
print("Custom Test Dataset Results:")
print("="*50)
for key, value in custom_metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")
print("="*50)


In [ ]:
# Generate detailed classification report
print("\nDetailed Classification Report for Custom Test Dataset:")
print("="*70)
report = classification_report(custom_labels, custom_predictions)
print(report)
print("="*70)


In [ ]:
# Display example predictions vs ground truth
print("\nExample Predictions vs Ground Truth:")
print("="*70)

# Show first 3 examples
num_examples = min(3, len(custom_notes))
for i in range(num_examples):
    print(f"\nExample {i+1}:")
    print(f"  Note: {custom_notes[i][:150]}...")
    print(f"  Words: {custom_notes[i].split()[:15]}...")
    print(f"  Ground Truth Tags: {custom_tags_filtered[i][:15]}...")
    print(f"  Predicted Tags: {custom_predictions[i][:15]}...")
    
    # Show matches/mismatches
    matches = sum(1 for gt, pred in zip(custom_labels[i][:15], custom_predictions[i][:15]) if gt == pred)
    total = min(15, len(custom_labels[i]))
    print(f"  Accuracy (first 15 tags): {matches}/{total} = {matches/total:.2%}")

print("\n" + "="*70)
